In [14]:
import pandas as pd
import numpy as np
from workalendar.america import BrazilRioDeJaneiro

# ══════════════════════════════════════════════════════════════════════════════
# Leitura 
# ══════════════════════════════════════════════════════════════════════════════

dfbruto = pd.read_json("dados brutos/linha_913.jsonl", lines=True)


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# Limpeza dos dados de GPS
# ══════════════════════════════════════════════════════════════════════════════

df_limpo = dfbruto.copy()

# Corrige vírgula decimal nas coordenadas
df_limpo["latitude"]   = df_limpo["latitude"].astype(str).str.replace(",", ".").astype(float)
df_limpo["longitude"]  = df_limpo["longitude"].astype(str).str.replace(",", ".").astype(float)
df_limpo["velocidade"] = pd.to_numeric(df_limpo["velocidade"], errors="coerce")

# Converte Unix ms → datetime no fuso de Brasília
df_limpo["datahora"] = pd.to_datetime(df_limpo["datahora"].astype(float), unit="ms", utc=True) \
                        .dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)

# Mantém só as colunas que importam
df_limpo = df_limpo[["ordem", "latitude", "longitude", "velocidade", "datahora", "linha"]]

# Filtros de qualidade:
df_limpo = df_limpo[
    df_limpo["latitude"].between(-23.10, -22.74) &
    df_limpo["longitude"].between(-43.80, -43.10) &
    df_limpo["velocidade"].between(0, 100)
]

# Remove pings duplicados do mesmo ônibus no mesmo instante
df_limpo = df_limpo.drop_duplicates(subset=["ordem", "datahora"])
df_limpo = df_limpo.sort_values(["ordem", "datahora"]).reset_index(drop=True)

# Depois de termos usado velocidade para limpar o dataset, podemos remover essa coluna
df_limpo = df_limpo.drop(columns=["velocidade"])

print(f"GPS limpo: {len(df_limpo):,} registros | {df_limpo['datahora'].min()} → {df_limpo['datahora'].max()}")
df_limpo.head(5)

GPS limpo: 2,005,795 registros | 2025-05-31 23:57:12 → 2026-06-01 00:59:44


,ordem,latitude,longitude,datahora,linha
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# Leitura e limpeza dos dados climáticos
# ══════════════════════════════════════════════════════════════════════════════

df_limpo_clima = df_limpo.copy()

def ler_climatempo(path):
    clima = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8-sig", quotechar='"', na_values=["", " "])
    clima = clima.rename(columns={
        "Data"           : "data",
        "Hora (UTC)"     : "hora_utc",
        "Temp. Max. (C)" : "temp_max",
        "Temp. Min. (C)" : "temp_min",
        "Chuva (mm)"     : "chuva",
    })
    clima = clima[["data", "hora_utc", "temp_max", "temp_min", "chuva"]]

    # Hora vem como "0000", "0100"... → converte para "00:00", "01:00"...
    clima["hora_utc"] = clima["hora_utc"].astype(str).str.zfill(4)
    clima["hora_utc"] = clima["hora_utc"].str[:2] + ":" + clima["hora_utc"].str[2:]

    # Constrói datetime em UTC
    clima["datahora_clima"] = pd.to_datetime(
        clima["data"] + " " + clima["hora_utc"],
        format="%d/%m/%Y %H:%M", utc=True
    ).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)

    return clima.drop(columns=["data", "hora_utc"]).sort_values("datahora_clima")

# Une os dois períodos em um único DataFrame climático
clima = pd.concat([
    ler_climatempo("dados brutos/climatempo 01-06-25--01-01-26.csv"),
    ler_climatempo("dados brutos/climatempo 02-01-26--01-06-26.csv"),
], ignore_index=True).drop_duplicates("datahora_clima")

print(f"Clima: {len(clima):,} registros | {clima['datahora_clima'].min()} → {clima['datahora_clima'].max()}")

# ══════════════════════════════════════════════════════════════════════════════
# Fazemos o join do clima
# ══════════════════════════════════════════════════════════════════════════════

df_limpo_clima = pd.merge_asof(
    df_limpo_clima.sort_values("datahora"),
    clima.rename(columns={"datahora_clima": "datahora"}),
    on="datahora",
    direction="nearest",
    tolerance=pd.Timedelta("1h")
)
df_limpo_clima = df_limpo_clima.sort_values(["ordem", "datahora"]).reset_index(drop=True)
df_limpo_clima.head(5)

Clima: 8,784 registros | 2025-05-31 21:00:00 → 2026-06-01 20:00:00


,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913,38.5,34.7,0.0
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913,38.5,34.7,0.0


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# Adicionamos features de tempo
# ══════════════════════════════════════════════════════════════════════════════

df_limpo_clima_feriados = df_limpo_clima.copy()

def adicionar_features_temporais(df, coluna="datahora"):
    dt = df[coluna]

    # Aprender hora de rush
    df["hora"]       = dt.dt.hour
    df["dia_semana"] = dt.dt.dayofweek  # 0=seg ... 6=dom
    df["fim_semana"] = (dt.dt.dayofweek >= 5).astype(int)

    # Feriados do Rio: nacionais + estaduais + municipais
    cal = BrazilRioDeJaneiro()
    feriados = set()
    for ano in dt.dt.year.unique():
        for data, _ in cal.holidays(int(ano)):
            feriados.add(data)

    df["feriado"]  = dt.dt.date.apply(lambda d: int(d in feriados))
    df["dia_util"] = ((df["fim_semana"] == 0) & (df["feriado"] == 0)).astype(int)

    return df

df_limpo_clima_feriados = adicionar_features_temporais(df_limpo_clima_feriados)
df_limpo_clima_feriados.head(5)

,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva,hora,dia_semana,fim_semana,feriado,dia_util
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0,13,1,0,0,1
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0,13,1,0,0,1
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0,13,1,0,0,1
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913,38.5,34.7,0.0,13,1,0,0,1
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913,38.5,34.7,0.0,13,1,0,0,1


In [18]:
# ═══════════════════════════════════════════════
# Vamos tirar linhas nulas
# ═══════════════════════════════════════════════
df_final = df_limpo_clima_feriados.copy()

# Vamos contar quantas linhas nulas temos e decidir se vamos tirar ou não
df_final.isnull().sum()/df_final.shape[0]*100

ordem         0.000000
latitude      0.000000
longitude     0.000000
datahora      0.000000
linha         0.000000
temp_max      0.030512
temp_min      0.030512
chuva         0.070695
hora          0.000000
dia_semana    0.000000
fim_semana    0.000000
feriado       0.000000
dia_util      0.000000
dtype: float64

In [19]:
# Temos pouco mais de 1% de linhas nulas, então vamos tirar essas linhas
n_linhas_nulas = df_final.isnull().any(axis=1).sum()
pct_linhas_nulas = n_linhas_nulas / len(df_final) * 100

print(f'Linhas com pelo menos um NaN: {n_linhas_nulas}')
print(f'Percentual: {pct_linhas_nulas:.4f}%')

Linhas com pelo menos um NaN: 1886
Percentual: 0.0940%


In [20]:
df_final.to_csv("saida/linha_913_final.csv", index=False)
print("\nSalvo em saida/linha_913_final.csv")


Salvo em saida/linha_913_final.csv
